# ML Deep Learning - LSTM/GRU pour Trading

**Objectif**: Utiliser les réseaux de neurones récurrents (LSTM/GRU) pour prédire les rendements futurs des actions.

## Stratégie

1. **Séquences temporelles**: Utiliser N jours passés pour prédire le rendement futur
2. **Architecture LSTM**: Capturer les dépendances à long terme dans les données
3. **Features multivariées**: Prix, volume, indicateurs techniques comme input
4. **Walk-forward training**: Ré-entraînement périodique pour éviter l'overfitting
5. **Gestion du risque**: Position sizing basé sur la confiance du modèle

## Prérequis

- tensorflow: `pip install tensorflow` (pour GPU: `tensorflow-gpu`)
- Alternative: pytorch, sklearn (pour prototypage rapide)
- Compréhension des RNN, LSTM, GRU

## Durée estimée: 60 minutes

In [ ]:
# Initialisation QuantBook
from AlgorithmImports import *

qb = QuantBook()

# Période d'analyse
qb.SetStartDate(2020, 1, 1)
qb.SetEndDate(2024, 12, 31)

print(f"Période: {qb.StartDate} à {qb.EndDate}")

## 1. Chargement des Données

In [ ]:
# Ajouter les indices ETF
tickers = ['SPY', 'QQQ', 'IWM', 'TLT']
symbols = {}

for ticker in tickers:
    symbols[ticker] = qb.AddEquity(ticker, Resolution.DAILY).Symbol

# Récupérer les données
history = qb.History(list(symbols.values()), 365*5, Resolution.Daily)

closes = history['close'].unstack(level=0)
volumes = history['volume'].unstack(level=0)

print(f"Données: {closes.shape[0]} jours, {closes.shape[1]} actifs")
closes.head()

## 2. Feature Engineering pour LSTM

In [ ]:
def create_lstm_features(closes, volumes, lookback=20):
    """Crée les features pour l'entraînement LSTM."""
    features = pd.DataFrame()
    
    for ticker in closes.columns:
        close = closes[ticker]
        volume = volumes[ticker]
        
        # Returns
        returns = close.pct_change()
        
        # Log returns
        log_returns = np.log(close / close.shift(1))
        
        # Price momentum
        momentum_5 = close / close.shift(5) - 1
        momentum_10 = close / close.shift(10) - 1
        momentum_20 = close / close.shift(20) - 1
        
        # Volatility
        volatility = returns.rolling(lookback).std()
        
        # RSI
        delta = close.diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        
        # Volume change
        volume_change = volume.pct_change()
        
        # Combiner les features
        ticker_features = pd.DataFrame({
            f'{ticker}_returns': returns,
            f'{ticker}_log_returns': log_returns,
            f'{ticker}_mom_5': momentum_5,
            f'{ticker}_mom_10': momentum_10,
            f'{ticker}_mom_20': momentum_20,
            f'{ticker}_volatility': volatility,
            f'{ticker}_rsi': rsi,
            f'{ticker}_volume_change': volume_change
        })
        
        features = pd.concat([features, ticker_features], axis=1)
    
    return features.fillna(0)

# Créer les features
features = create_lstm_features(closes, volumes)

print(f"Features shape: {features.shape}")
print(f"\nColonnes de features:")
print(features.columns.tolist())

## 3. Préparation des Séquences

In [ ]:
def create_sequences(features, targets, sequence_length=20):
    """Crée des séquences pour l'entraînement LSTM.
    
    Args:
        features: DataFrame des features
        targets: Série des cibles (rendements futurs)
        sequence_length: Longueur de la séquence
    
    Returns:
        X: Array des séquences (samples, sequence_length, features)
        y: Array des cibles
    """
    X, y = [], []
    
    for i in range(sequence_length, len(features)):
        # Séquence de features
        sequence = features.iloc[i-sequence_length:i].values
        X.append(sequence)
        
        # Cible: rendement futur
        y.append(targets.iloc[i])
    
    return np.array(X), np.array(y)

# Préparer les données pour SPY
spy_returns = closes['SPY'].pct_change().fillna(0)
spy_features = features[[c for c in features.columns if 'SPY' in c]]

# Créer les séquences
sequence_length = 20
X, y = create_sequences(spy_features, spy_returns, sequence_length)

print(f"Séquences X shape: {X.shape}")
print(f"Cibles y shape: {y.shape}")
print(f"\nDétails:")
print(f"  - Samples: {X.shape[0]}")
print(f"  - Time steps: {X.shape[1]}")
print(f"  - Features: {X.shape[2]}")

## 4. Normalisation des Données

In [ ]:
from sklearn.preprocessing import StandardScaler

def normalize_sequences(X, y, fit_on_train=True, scaler=None):
    """Normalise les séquences pour le LSTM."""
    
    # Reshape pour scaler (samples * timesteps, features)
    original_shape = X.shape
    X_reshaped = X.reshape(-1, X.shape[-1])
    
    if scaler is None:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_reshaped)
    else:
        X_scaled = scaler.transform(X_reshaped)
    
    # Reshape back
    X_scaled = X_scaled.reshape(original_shape)
    
    return X_scaled, y, scaler

# Normaliser
X_normalized, y_normalized, scaler = normalize_sequences(X, y)

print(f"X normalisé shape: {X_normalized.shape}")
print(f"Moyenne par feature: {X_normalized.mean(axis=0)[0].round(3)}")
print(f"Std par feature: {X_normalized.std(axis=0)[0].round(3)}")

## 5. Architecture LSTM

In [ ]:
# Note: En production, utilisez TensorFlow/Keras ou PyTorch
# Ici, nous utilisons sklearn pour le prototypage rapide

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

def build_lstm_model(input_shape):
    """Construit un modèle LSTM.
    
    En production avec TensorFlow:
    ```python
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    
    model.compile(optimizer='adam', loss='mse')
    return model
    ```
    """
    # Simplifié: utiliser Ridge comme proxy
    # Flatten les séquences
    return Ridge(alpha=1.0)

def train_model(X_train, y_train, X_test, y_test):
    """Entraîne le modèle."""
    # Flatten pour Ridge
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    
    model = Ridge(alpha=1.0)
    model.fit(X_train_flat, y_train)
    
    # Évaluer
    train_pred = model.predict(X_train_flat)
    test_pred = model.predict(X_test_flat)
    
    train_mse = ((y_train - train_pred) ** 2).mean()
    test_mse = ((y_test - test_pred) ** 2).mean()
    
    return model, train_mse, test_mse

# Split train/test
split_idx = int(len(X_normalized) * 0.8)

X_train, X_test = X_normalized[:split_idx], X_normalized[split_idx:]
y_train, y_test = y_normalized[:split_idx], y_normalized[split_idx:]

# Entraîner
model, train_mse, test_mse = train_model(X_train, y_train, X_test, y_test)

print(f"Train MSE: {train_mse:.6f}")
print(f"Test MSE: {test_mse:.6f}")

## 6. Prédictions et Analyse

In [ ]:
# Faire des prédictions
X_test_flat = X_test.reshape(X_test.shape[0], -1)
predictions = model.predict(X_test_flat)

# Analyser les résultats
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': predictions,
    'Abs_Error': np.abs(y_test - predictions)
})

print("Statistiques des prédictions:")
print(results.describe())

# Direction accuracy
direction_correct = ((results['Actual'] > 0) == (results['Predicted'] > 0)).mean()
print(f"\nAccuracy de direction: {direction_correct:.2%}")

Importation des bibliothèques d'analyse de données et de visualisation.

In [ ]:
import matplotlib.pyplot as plt

# Visualiser les prédictions vs réel
plt.figure(figsize=(14, 6))

# Premier 50 points
n_plot = 50
plt.plot(results['Actual'].iloc[:n_plot].values, label='Réel', marker='o')
plt.plot(results['Predicted'].iloc[:n_plot].values, label='Prédit', marker='x')

plt.xlabel('Time')
plt.ylabel('Return')
plt.title('Prédictions LSTM vs Réel (SPY)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Walk-Forward Validation

In [ ]:
def walk_forward_validation(features, targets, train_size=252, test_size=63):
    """Validation walk-forward pour simuler le trading réel."""
    
    predictions = []
    actuals = []
    
    for start in range(0, len(features) - train_size - test_size, test_size):
        train_end = start + train_size
        test_end = train_end + test_size
        
        # Données d'entraînement
        X_train, y_train = create_sequences(
            features.iloc[start:train_end],
            targets.iloc[start:train_end],
            sequence_length
        )
        
        # Normaliser
        X_train_norm, y_train_norm, scaler = normalize_sequences(X_train, y_train)
        
        # Entraîner
        X_train_flat = X_train_norm.reshape(X_train_norm.shape[0], -1)
        model = Ridge(alpha=1.0)
        model.fit(X_train_flat, y_train_norm)
        
        # Prédire sur test set
        X_test, y_test = create_sequences(
            features.iloc[train_end:test_end],
            targets.iloc[train_end:test_end],
            sequence_length
        )
        
        X_test_norm, _, _ = normalize_sequences(X_test, y_test, scaler=scaler)
        X_test_flat = X_test_norm.reshape(X_test_norm.shape[0], -1)
        
        pred = model.predict(X_test_flat)
        
        predictions.extend(pred)
        actuals.extend(y_test.values)
    
    return np.array(predictions), np.array(actuals)

# Exécuter walk-forward
wf_pred, wf_actual = walk_forward_validation(spy_features, spy_returns)

# Évaluer
wf_mse = ((wf_actual - wf_pred) ** 2).mean()
wf_direction = ((wf_actual > 0) == (wf_pred > 0)).mean()

print(f"Walk-Forward Results:")
print(f"MSE: {wf_mse:.6f}")
print(f"Direction Accuracy: {wf_direction:.2%}")

## 8. Backtest de la Stratégie LSTM

In [ ]:
def backtest_lstm_strategy(closes, volumes, features, 
                          train_period=252, rebalance_freq=5):
    """Backtest la stratégie LSTM."""
    
    portfolio_value = 100000
    positions = {}
    
    trade_dates = []
    
    # Commencer après la période d'entraînement initiale
    for i in range(train_period, len(closes) - sequence_length, rebalance_freq):
        current_date = closes.index[i]
        trade_dates.append(current_date)
        
        # Entraîner sur données passées
        train_features = features.iloc[i-train_period:i]
        train_returns = closes['SPY'].pct_change().iloc[i-train_period:i]
        
        X_train, y_train = create_sequences(train_features, train_returns, sequence_length)
        
        if len(X_train) < 10:
            continue
        
        X_train_norm, y_train_norm, scaler = normalize_sequences(X_train, y_train)
        X_train_flat = X_train_norm.reshape(X_train_norm.shape[0], -1)
        
        model = Ridge(alpha=1.0)
        model.fit(X_train_flat, y_train_norm)
        
        # Prédire pour chaque ETF
        predictions = {}
        for ticker in tickers:
            ticker_features = features[[c for c in features.columns if ticker in c]]
            ticker_returns = closes[ticker].pct_change()
            
            if len(ticker_features) < i + sequence_length:
                continue
            
            recent_features = ticker_features.iloc[i-sequence_length:i]
            
            X_pred = recent_features.values.reshape(1, sequence_length, -1)
            X_pred_norm, _, _ = normalize_sequences(X_pred, np.array([0]), scaler=scaler)
            X_pred_flat = X_pred_norm.reshape(1, -1)
            
            pred = model.predict(X_pred_flat)[0]
            predictions[ticker] = pred
        
        # Vendre positions existantes
        for ticker, qty in positions.items():
            portfolio_value += qty * closes[ticker].iloc[i]
        
        positions = {}
        
        # Acheter top 2 prédictions positives
        sorted_preds = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
        
        position_size = portfolio_value / 2
        count = 0
        
        for ticker, pred_return in sorted_preds:
            if pred_return > 0 and count < 2:
                buy_price = closes[ticker].iloc[i]
                qty = position_size / buy_price
                positions[ticker] = qty
                portfolio_value -= qty * buy_price
                count += 1
    
    # Valeur finale
    final_value = portfolio_value
    for ticker, qty in positions.items():
        final_value += qty * closes[ticker].iloc[-1]
    
    return {
        'initial': 100000,
        'final': final_value,
        'return': (final_value - 100000) / 100000,
        'dates': trade_dates
    }

# Exécuter le backtest
backtest_results = backtest_lstm_strategy(closes, volumes, features)

print(f"\nBacktest Results:")
print(f"Valeur initiale: ${backtest_results['initial']:,.2f}")
print(f"Valeur finale: ${backtest_results['final']:,.2f}")
print(f"Return total: {backtest_results['return']:.2%}")

## 9. Architecture GRU Alternative

In [ ]:
# GRU (Gated Recurrent Unit) est une alternative plus légère au LSTM
# En production avec TensorFlow:

gru_architecture = '''
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

model = Sequential([
    GRU(64, return_sequences=True, input_shape=(sequence_length, n_features)),
    Dropout(0.2),
    GRU(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
# model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)
'''

print("Architecture GRU ( TensorFlow ):")
print(gru_architecture)

print("\nAvantages GRU vs LSTM:")
print("- Moins de paramètres (plus rapide à entraîner)")
print("- Performance similaire pour beaucoup de tâches")
print("- Meilleur pour les séquences plus courtes")

## 10. Améliorations et Extensions

In [ ]:
# Idées d'amélioration:

improvements = {
    'Architecture': [
        'Bidirectional LSTM: capture les dépendances passées et futures',
        'Attention mechanism: focus sur les parties importantes de la séquence',
        'Transformer: self-attention pour longues dépendances'
    ],
    'Features': [
        'Market sentiment (VIX, put/call ratio)',
        'Macro indicators (rates, GDP, inflation)',
        'Alternative data (satellite, credit card, web scraping)'
    ],
    'Training': [
        'Transfer learning: pré-entraîner sur données générales',
        'Data augmentation: bruit, jittering, scaling',
        'Hyperparameter optimization: Bayesian, genetic algorithms'
    ],
    'Deployment': [
        'ONNX runtime: inference optimisée',
        'Quantization: INT8 pour edge deployment',
        'Pruning: supprimer les connexions inutiles'
    ]
}

for category, items in improvements.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  - {item}")